# Textual transformations and detection loss

Minimal, reproducible analysis for H2L (`human → h2l`) and LLM2L (`free_llm → llm2l`). It uses the project `ScoreAnalyzer`/`TextAnalyzer`, detector NPZ metadata, block-bootstrap confidence intervals, cluster-bootstrap Spearman intervals, and OLS with HC3 robust standard errors.

Before the first run, create the non-destructive aligned artifacts with `venv/bin/python scripts/realign_h2l.py`. The notebook intentionally reads `results-aligned`, never the legacy positionally misaligned H2L results.

`perplexity_proxy` is explicitly a lightweight **compression-surprisal shift** (output minus source compressed bits/token), not neural-LM perplexity. SBERT cosine similarity is the primary semantic metric. BERTScore is not run because it is optional and not installed in this environment.

In [1]:
from pathlib import Path
import zlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

from utils.evaluation import Metrics, ScoreAnalyzer, TextAnalyzer
from utils.evaluation.ResultsReportGraphBuilder import ResultsReportGraphBuilder
from utils.evaluation.TextAnalyzer import edit_tokens

# Use the non-destructive H2L-aligned migration produced by
# `venv/bin/python scripts/realign_h2l.py`.
RESULTS_DIR = Path("results-aligned")
OUTPUT_DIR = RESULTS_DIR / "text_analysis"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
N_BOOTSTRAP = 5000
ALPHA = 0.05
SEMANTIC_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
TEXT_CACHE = OUTPUT_DIR / "text_pair_metrics.parquet"

sns.set_theme(style="whitegrid", context="paper")

/home/gx1/git/when-human-write-like-llm/when-human-write-like-machines/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Build source-output pairs and text metrics

The first uncached run loads SBERT and can take time. Later runs reuse the Parquet cache. Delete the cache if inputs or metric definitions change.

In [5]:
if TEXT_CACHE.exists():
    text_pairs = pd.read_parquet(TEXT_CACHE)
else:
    metrics = Metrics.load_from_folder(RESULTS_DIR)
    score_analyzer = ScoreAnalyzer(metrics)
    text_analyzer = TextAnalyzer(
        score_analyzer,
        semantic_model_name=SEMANTIC_MODEL,
        batch_size=32,
        show_progress_bar=True,
    )
    text_pairs = text_analyzer.to_dataframe()
    text_pairs.to_parquet(TEXT_CACHE, index=False)

print(text_pairs.shape)
text_pairs.groupby(["source_regime", "rewritten_regime"]).size()

(14400, 16)


source_regime  rewritten_regime
free_llm       llm2l               7200
human          h2l                 7200
dtype: int64

In [6]:
def compressed_bits_per_token(text):
    token_count = len(edit_tokens(text))
    if token_count == 0:
        return np.nan
    return 8 * len(zlib.compress(text.encode("utf-8"), level=9)) / token_count

# Positive values mean that the output is less compressible per token.
text_pairs["source_compression_bits_per_token"] = text_pairs["source_text"].map(compressed_bits_per_token)
text_pairs["output_compression_bits_per_token"] = text_pairs["rewritten_text"].map(compressed_bits_per_token)
text_pairs["perplexity_proxy"] = (
    text_pairs["output_compression_bits_per_token"]
    - text_pairs["source_compression_bits_per_token"]
)

TEXT_FEATURES = [
    "word_ratio",
    "normalized_token_edit_distance",
    "lexical_jaccard",
    "semantic_similarity",
    "perplexity_proxy",
]
text_pairs.groupby("rewritten_regime")[TEXT_FEATURES].describe().round(3)

word_ratio                                                   \
                      count   mean    std    min    25%    50%    75%    max   
rewritten_regime                                                               
h2l                  7200.0  0.798  0.169  0.023  0.699  0.814  0.915  1.865   
llm2l                7200.0  0.887  0.115  0.111  0.833  0.909  0.963  3.060   

                 normalized_token_edit_distance         ...  \
                                          count   mean  ...   
rewritten_regime                                        ...   
h2l                                      7200.0  0.602  ...   
llm2l                                    7200.0  0.348  ...   

                 semantic_similarity        perplexity_proxy                \
                                 75%    max            count   mean    std   
rewritten_regime                                                             
h2l                            0.938  0.994           7200.0  1.799  1.837   
llm2l                          0.976  1.000           7200.0  0.925  3.298   

                                                        
                     min    25%    50%    75%      max  
rewritten_regime                                        
h2l              -13.294  0.735  1.577  2.634   45.201  
llm2l            -50.913  0.250  0.733  1.341  130.728  

[2 rows x 40 columns]

## 2. H2L vs LLM2L table

Confidence intervals resample the 12 dataset×generator blocks, rather than treating all texts as independent experimental units.

In [8]:
def block_bootstrap_mean_ci(frame, feature, rng, n_bootstrap=N_BOOTSTRAP, alpha=ALPHA):
    block_means = (
        frame.groupby(["dataset", "generator"], observed=True)[feature]
        .mean()
        .dropna()
        .to_numpy()
    )
    if len(block_means) == 0:
        return np.nan, np.nan, np.nan
    samples = rng.choice(block_means, size=(n_bootstrap, len(block_means)), replace=True).mean(axis=1)
    low, high = np.quantile(samples, [alpha / 2, 1 - alpha / 2])
    return block_means.mean(), low, high

REGIME_LABELS = {"h2l": "H2L", "llm2l": "LLM2L"}
SUMMARY_FEATURES = TEXT_FEATURES[:4]
rng = np.random.default_rng(RANDOM_SEED)
summary_rows = []
for regime, label in REGIME_LABELS.items():
    regime_frame = text_pairs[text_pairs["rewritten_regime"] == regime]
    for feature in SUMMARY_FEATURES:
        mean, low, high = block_bootstrap_mean_ci(regime_frame, feature, rng)
        summary_rows.append({"Regime": label, "Feature": feature, "mean": mean, "ci_low": low, "ci_high": high})

text_summary_numeric = pd.DataFrame(summary_rows)
text_summary_table = text_summary_numeric.assign(
    value=lambda x: x.apply(lambda r: f"{r['mean']:.3f} [{r['ci_low']:.3f}, {r['ci_high']:.3f}]", axis=1)
).pivot(index="Regime", columns="Feature", values="value").loc[["H2L", "LLM2L"], SUMMARY_FEATURES]
text_summary_table.columns = ["Word ratio", "NED token", "Jaccard", "Semantic similarity"]
text_summary_table.to_csv(OUTPUT_DIR / "h2l_vs_llm2l_text_metrics.csv")
text_summary_table

,Word ratio,NED token,Jaccard,Semantic similarity
Regime,,,,
H2L,"0.798 [0.746, 0.845]","0.602 [0.575, 0.628]","0.463 [0.436, 0.489]","0.886 [0.866, 0.904]"
LLM2L,"0.887 [0.857, 0.916]","0.348 [0.301, 0.395]","0.587 [0.534, 0.648]","0.949 [0.939, 0.958]"


## 3. Detection loss at detector×dataset×generator×regime level

`Detection loss = TPR_FreeLLM − TPR_target`. Text features are detector-independent block means and are joined to every detector evaluated on that block.

In [9]:
result_rows = ResultsReportGraphBuilder(RESULTS_DIR).load_npz_metrics()
block_keys = ["detector", "dataset", "generator"]
result_rows.groupby("target_regime").size()

target_regime
free_llm    60
h2l         60
llm2l       60
dtype: int64

In [10]:
baseline = result_rows.loc[
    result_rows["target_regime"] == "free_llm",
    block_keys + ["tpr_at_fpr"],
].rename(columns={"tpr_at_fpr": "tpr_free_llm"})
targets = result_rows.loc[
    result_rows["target_regime"].isin(["h2l", "llm2l"]),
    block_keys + ["target_regime", "tpr_at_fpr"],
].rename(columns={"tpr_at_fpr": "tpr_target"})

detection = targets.merge(baseline, on=block_keys, how="inner", validate="many_to_one")
detection["detection_loss"] = detection["tpr_free_llm"] - detection["tpr_target"]

text_blocks = (
    text_pairs.groupby(["dataset", "generator", "rewritten_regime"], observed=True)[TEXT_FEATURES]
    .mean()
    .reset_index()
    .rename(columns={"rewritten_regime": "target_regime"})
)
analysis_frame = detection.merge(
    text_blocks,
    on=["dataset", "generator", "target_regime"],
    how="inner",
    validate="many_to_one",
)
analysis_frame.to_csv(OUTPUT_DIR / "detection_loss_with_text_features.csv", index=False)
print(analysis_frame.shape)
analysis_frame.head()

(120, 12)


,detector,dataset,generator,target_regime,tpr_target,tpr_free_llm,detection_loss,word_ratio,normalized_token_edit_distance,lexical_jaccard,semantic_similarity,perplexity_proxy
0,BERT-Defense,owt,gemma2_9b,h2l,0.005000,0.000000,-0.005000,0.640219,0.656944,0.429918,0.879530,2.383870
1,BERT-Defense,owt,gemma2_9b,llm2l,0.003333,0.000000,-0.003333,0.800743,0.445225,0.532515,0.948187,0.842749
2,BERT-Defense,owt,llama32_3b,h2l,0.005000,0.000000,-0.005000,0.780852,0.622866,0.483550,0.902141,1.216375
3,BERT-Defense,owt,llama32_3b,llm2l,0.000000,0.000000,0.000000,0.884834,0.265061,0.745971,0.966904,0.955892
4,BERT-Defense,owt,mistral7b,h2l,0.015000,0.001667,-0.013333,0.851070,0.588077,0.474803,0.910444,1.184600


## 4. Spearman correlation with detection loss

The point estimate and p-value use Spearman's test. The CI uses a cluster bootstrap over dataset×generator×regime blocks, keeping all detector observations from a sampled block together.

In [11]:
def clustered_spearman_ci(frame, feature, rng, n_bootstrap=N_BOOTSTRAP, alpha=ALPHA):
    clean = frame.dropna(subset=[feature, "detection_loss"])
    rho, p_value = stats.spearmanr(clean[feature], clean["detection_loss"])
    groups = [g for _, g in clean.groupby(["dataset", "generator", "target_regime"], observed=True)]
    boot = np.empty(n_bootstrap)
    for i in range(n_bootstrap):
        sampled = pd.concat([groups[j] for j in rng.integers(0, len(groups), len(groups))], ignore_index=True)
        boot[i] = stats.spearmanr(sampled[feature], sampled["detection_loss"]).statistic
    low, high = np.nanquantile(boot, [alpha / 2, 1 - alpha / 2])
    return rho, low, high, p_value

FEATURE_LABELS = {
    "normalized_token_edit_distance": "NED token",
    "semantic_similarity": "Semantic similarity",
    "lexical_jaccard": "Jaccard",
    "word_ratio": "Word ratio",
    "perplexity_proxy": "Compression-surprisal shift (proxy)",
}
rng = np.random.default_rng(RANDOM_SEED)
correlation_rows = []
for feature, label in FEATURE_LABELS.items():
    rho, low, high, p_value = clustered_spearman_ci(analysis_frame, feature, rng)
    correlation_rows.append({"Feature": label, "Spearman rho": rho, "CI low": low, "CI high": high, "p-value": p_value})

correlation_table = pd.DataFrame(correlation_rows)
correlation_table.to_csv(OUTPUT_DIR / "spearman_detection_loss.csv", index=False)
correlation_table.round(3)

,Feature,Spearman rho,CI low,CI high,p-value
0,NED token,0.186,0.107,0.242,0.042
1,Semantic similarity,-0.096,-0.164,-0.009,0.297
2,Jaccard,-0.146,-0.206,-0.052,0.111
3,Word ratio,-0.133,-0.212,-0.017,0.147
4,Compression-surprisal shift (proxy),0.180,0.094,0.243,0.049


## 5. Explanatory regression

OLS specification: `DetectionLoss ~ NED + SemanticSimilarity + WordRatio + Dataset + Generator + Detector`. Numeric predictors are standardized, categorical predictors use treatment coding, and coefficient inference uses HC3 heteroskedasticity-robust standard errors.

In [12]:
def fit_ols_hc3(frame):
    numeric = ["normalized_token_edit_distance", "semantic_similarity", "word_ratio"]
    categorical = ["dataset", "generator", "detector"]
    model_data = frame[["detection_loss"] + numeric + categorical].dropna().copy()

    standardized = (model_data[numeric] - model_data[numeric].mean()) / model_data[numeric].std(ddof=0)
    standardized.columns = ["NED token (z)", "Semantic similarity (z)", "Word ratio (z)"]
    dummies = pd.get_dummies(model_data[categorical], prefix=categorical, drop_first=True, dtype=float)
    design = pd.concat([standardized, dummies], axis=1)
    design.insert(0, "Intercept", 1.0)

    X = design.to_numpy(dtype=float)
    y = model_data["detection_loss"].to_numpy(dtype=float)
    xtx_inv = np.linalg.pinv(X.T @ X)
    beta = xtx_inv @ X.T @ y
    residual = y - X @ beta
    leverage = np.einsum("ij,jk,ik->i", X, xtx_inv, X)
    hc3_residual = residual / np.clip(1 - leverage, 1e-12, None)
    meat = (X * hc3_residual[:, None]).T @ (X * hc3_residual[:, None])
    covariance = xtx_inv @ meat @ xtx_inv
    std_error = np.sqrt(np.clip(np.diag(covariance), 0, None))

    df_resid = len(y) - np.linalg.matrix_rank(X)
    critical = stats.t.ppf(1 - ALPHA / 2, df_resid)
    t_stat = np.divide(beta, std_error, out=np.full_like(beta, np.nan), where=std_error > 0)
    p_value = 2 * stats.t.sf(np.abs(t_stat), df_resid)
    coefficients = pd.DataFrame({
        "term": design.columns,
        "coefficient": beta,
        "std_error_HC3": std_error,
        "ci_low": beta - critical * std_error,
        "ci_high": beta + critical * std_error,
        "p_value": p_value,
    })
    ss_residual = np.sum(residual ** 2)
    ss_total = np.sum((y - y.mean()) ** 2)
    r_squared = 1 - ss_residual / ss_total
    adjusted_r_squared = 1 - (1 - r_squared) * (len(y) - 1) / df_resid
    fit = pd.Series({"n": len(y), "R_squared": r_squared, "adjusted_R_squared": adjusted_r_squared, "df_resid": df_resid})
    return coefficients, fit

regression_coefficients, regression_fit = fit_ols_hc3(analysis_frame)
regression_coefficients.to_csv(OUTPUT_DIR / "regression_coefficients_hc3.csv", index=False)
regression_fit.to_csv(OUTPUT_DIR / "regression_fit.csv", header=["value"])
display(regression_fit.round(3))
regression_coefficients.round(3)

n                     120.000
R_squared               0.626
adjusted_R_squared      0.584
df_resid              107.000
dtype: float64

,term,coefficient,std_error_HC3,ci_low,ci_high,p_value
0,Intercept,-0.136,0.077,-0.289,0.017,0.081
1,NED token (z),0.143,0.078,-0.011,0.297,0.069
2,Semantic similarity (z),0.031,0.091,-0.149,0.211,0.735
3,Word ratio (z),-0.084,0.041,-0.166,-0.001,0.047
4,dataset_wp,0.046,0.121,-0.194,0.287,0.703
5,dataset_xsum,0.089,0.049,-0.008,0.186,0.073
6,generator_llama32_3b,0.096,0.070,-0.043,0.235,0.173
7,generator_mistral7b,0.220,0.089,0.043,0.398,0.015
8,generator_qwen25_7b,0.027,0.065,-0.102,0.157,0.675
9,detector_FastDetectGPT,0.366,0.055,0.257,0.474,0.000


## Reading the outputs

- High semantic similarity with moderate NED supports surface rewriting with content preservation.
- Similar NED for H2L and LLM2L but different detection loss argues against modification quantity as the sole explanation.
- A Spearman CI excluding zero supports a monotonic association with detection loss; it does not establish causality.
- Standardized regression coefficients compare the three textual predictors on a common scale. Remaining dataset/generator/detector effects support the claim that text features explain only part of the loss.
- Treat the compression measure only as a proxy. For a final neural-perplexity robustness check, replace it with a preregistered causal LM and report the model/checkpoint and truncation policy.